# PSTT vs 通常 Gemma-3 Attention ヒートマップ比較＆ARC-AGI-3 連携版

**バージョン: 28a7e3b** (最終更新: 2026-04-10 18:32:15 JST)

このノートブックは、以下を実装しています：

1. **PSTT アーキテクチャ**
   - SigLIP最終層のattentionから高次重要度を抽出
   - べき集合パイプラインで複合パッチトークンを生成
   - Gemmaの視覚処理を強化

2. **SigLIP Attention統合**
   - 各パッチへの「受け取るattention」合計で重要度判定
   - ノルムベースから認知的重要度へシフト

3. **ARC-AGI-3 API連携**
   - 実際のゲーム画像を取得して推論可能
   - frame_to_image()でARCグリッドをRGB画像に変換

## 使用方法

### パターン1: テスト画像で推論（API不要）
セル12を実行 → Control版とExperimental版を比較

### パターン2: ARC-AGI-3の実際のゲームで推論
```python
infer_arc_game(pspt_enabled=True)
```
※ `.env` に `ARC_API_KEY` を設定が必要

## メモリ最適化
- セル3でPSTT_ENABLEDを設定（True/False）
- False時は PSTT クラスとモデルをロードしない（省メモリ）

## セル 1: 環境セットアップ

In [ ]:
# 必要なライブラリのインストール
!pip install -q transformers torch accelerate pillow huggingface_hub matplotlib scikit-image

# HuggingFace ログイン（Gemma3アクセスに必要）
from huggingface_hub import login
import os

# Colab のシークレットから HF_TOKEN を取得するか、直接入力
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✓ HuggingFace にログインしました")
    else:
        print("⚠️  HF_TOKEN が見つかりません。userdata.get('HF_TOKEN') を使用するか、手動で login() してください。")
except ImportError:
    # Colab以外の環境での実行
    print("⚠️  Colab環境ではありません。.envまたは環境変数からHF_TOKENを取得してください。")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PSTT モード設定（Control/Experimental）
# ═══════════════════════════════════════════════════════════════
# ⚠️  このセルはCell4, Cell6より前に実行してください

PSTT_ENABLED = False  # ← True/False で切り替え

print("\n" + "="*70)
print("🔧 PSTT モード設定")
print("="*70)
if PSTT_ENABLED:
    print("✓ PSTT モード: ON (Experimental)")
    print("  実行内容:")
    print("    • SigLIP最終層のattentionを取得")
    print("    • Step1: attention から top-10パッチを抽出")
    print("    • Step2-4: べき集合→スコアリング→Refinement")
    print("    • Gemmaに処理済みembeddingsを入力")
else:
    print("✗ PSTT モード: OFF (Control)")
    print("  実行内容:")
    print("    • SigLIP embeddingsを直接Gemmaに入力")
    print("    • 標準的なGemma-3推論のみ")
    print("    • PSTTによる工夫なし")
print("="*70 + "\n")

if PSTT_ENABLED:
    import torch
    import torch.nn as nn
    import numpy as np
    from typing import Optional, Tuple, List

    # ═══════════════════════════════════════════════════════════════
    # PSTT Module: Power-Set Tokenizer Transformer
    # ═══════════════════════════════════════════════════════════════

    class PSTTModule(nn.Module):
        """
        4段階のパイプライン:
        Step1: SigLIP最終層のattentionから上位10シードトークン抽出（高次重要度）
        Step2: Power-Set → 2^10-1=1023 グループトークン生成（べき集合）
        Step3: Logical Scoring → 上位20グループに絞り込み
        Step4: Gated Cross-Refinement → 元512パッチを論理文脈で強化
        """
        def __init__(self, embed_dim=768, num_heads=8, top_k_seeds=10, top_k_groups=20):
            super().__init__()
            self.embed_dim = embed_dim
            self.num_heads = num_heads
            self.top_k_seeds = top_k_seeds
            self.top_k_groups = top_k_groups

            # Step3: グループスコアリング用 Transformer
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4,
                batch_first=True, norm_first=True, dropout=0.1
            )
            self.group_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

            # Step4: Gated Cross-Refinement
            self.gate_proj = nn.Linear(embed_dim, embed_dim)
            self.refine_proj = nn.Linear(embed_dim * 2, embed_dim)

        def forward(self, x: torch.Tensor, siglip_attention: torch.Tensor = None) -> torch.Tensor:
            """
            Args:
                x: (batch, seq_len, embed_dim) - vision encoder output
                siglip_attention: (batch, num_heads, seq_len, seq_len) - SigLIP最終層のattention
                                  Noneの場合は、従来のノルムベースで処理
            Returns:
                (batch, seq_len, embed_dim) - refined token embeddings
            """
            B, N, D = x.shape

            # ──── Step 1: Pair-wise Scanning → Top-K Seeds ────
            if siglip_attention is not None:
                # SigLIPのattentionから重要度を計算
                # 各パッチjへの「受け取るattention」の合計 = 高次重要度
                # shape: (batch, num_heads, seq_len, seq_len)
                # → 各パッチがどれだけ他のパッチから注目されているか
                attn_in = siglip_attention.mean(dim=1)  # (batch, seq_len, seq_len) - ヘッド平均
                patch_importance = attn_in.sum(dim=1)  # (batch, seq_len) - 各パッチへの合計attention
            else:
                # フォールバック: ノルムベース
                patch_importance = torch.norm(x, dim=-1)  # (B, N)

            _, seed_indices = torch.topk(patch_importance, k=min(self.top_k_seeds, N), dim=1)

            # seed tokenの集約
            seed_tokens = torch.gather(x, 1, seed_indices.unsqueeze(-1).expand(-1, -1, D))  # (B, K, D)

            # ──── Step 2: Power-Set → Group Tokens ────
            # べき集合: 2^K 通りのすべての部分集合を生成
            K = seed_tokens.shape[1]
            group_tokens = []

            # 2^K - 1 通りの部分集合を列挙（空集合は除外）
            for mask in range(1, 2**K):
                subset_indices = []
                for i in range(K):
                    if (mask >> i) & 1:  # i番目のビットが1なら、Si を選択
                        subset_indices.append(i)

                # 部分集合の和：{Si, Sj, Sk, ...} → Si + Sj + Sk + ...
                # shape: (B, D)
                subset_sum = seed_tokens[:, subset_indices, :].sum(dim=1, keepdim=True)  # (B, 1, D)
                group_tokens.append(subset_sum)

            # すべての部分集合をcat
            group_tokens = torch.cat(group_tokens, dim=1)  # (B, 2^K-1, D)

            # ──── Step 3: Logical Scoring ────
            # Transformer でグループ間の関係を学習
            group_scored = self.group_transformer(group_tokens)  # (B, G, D)
            group_importance = torch.norm(group_scored, dim=-1)  # (B, G)
            _, top_group_idx = torch.topk(
                group_importance,
                k=min(self.top_k_groups, group_scored.shape[1]),
                dim=1
            )
            top_groups = torch.gather(group_scored, 1, top_group_idx.unsqueeze(-1).expand(-1, -1, D))  # (B, top_k, D)

            # ──── Step 4: Gated Cross-Refinement ────
            # top_groups を元の全トークン N に反映
            # グローバルコンテキスト
            global_context = top_groups.mean(dim=1, keepdim=True).expand(B, N, D)  # (B, N, D)

            # ゲーテッド融合
            gate = torch.sigmoid(self.gate_proj(global_context))  # (B, N, D)
            combined = torch.cat([x, global_context], dim=-1)  # (B, N, 2D)
            refined = self.refine_proj(combined)  # (B, N, D)

            # 残差接続
            output = x + gate * refined

            return output


    class PSTTVisionWrapper(nn.Module):
        """Vision tower（SigLIP）と multi_modal_projector の間に挿入するラッパー"""
        def __init__(self, embed_dim=768, num_heads=8):
            super().__init__()
            self.pstt = PSTTModule(embed_dim=embed_dim, num_heads=num_heads)
            self.siglip_attention = None  # SigLIPのattentionを保存

        def set_siglip_attention(self, attn):
            """SigLIPの最終層attentionを設定"""
            self.siglip_attention = attn

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.pstt(x, siglip_attention=self.siglip_attention)


    def install_pstt(model, freeze_base=True):
        """
        既存のモデルに PSTT を挿入する
        Args:
            model: AutoModelForCausalLM
            freeze_base: True の場合、ベースモデルを固定、PSTT のみ学習可能
        Returns:
            pstt_wrapper: 挿入された PSTT ラッパーモジュール
        """
        # Gemma-3 の場合、vision_tower の出力次元は 768
        embed_dim = 768

        # PSTT モジュール作成
        pstt_wrapper = PSTTVisionWrapper(embed_dim=embed_dim, num_heads=8)
        pstt_wrapper = pstt_wrapper.to(model.device)

        # multi_modal_projector の前処理として PSTT を実行
        if hasattr(model, 'multi_modal_projector'):
            original_forward = model.multi_modal_projector.forward

            def new_forward(x):
                x = pstt_wrapper(x)
                return original_forward(x)

            model.multi_modal_projector.forward = new_forward

        # ベースモデルを固定
        if freeze_base:
            for param in model.parameters():
                param.requires_grad = False
            # PSTT パラメータは学習可能に
            for param in pstt_wrapper.parameters():
                param.requires_grad = True

        print(f"✓ PSTT モジュールをモデルに挿入しました (embed_dim={embed_dim})")
        print(f"  注: SigLIPのattentionから重要度を判定します")
        return pstt_wrapper

    print("✓ PSTT アーキテクチャ定義完了")

else:
    print("ℹ️  PSTT モード OFF - PSTT クラス定義をスキップします")
    print("   モード ON に切り替えるには、上のセルで PSTT_ENABLED = True に設定してください")

from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-4b-it"
NUM_LAYERS = 34

print(f"Loading {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 通常版（常にロード）
print("[1/2] Loading base model (normal Gemma-3)...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model_base.eval()
print(f"✓ Base model loaded (device={next(model_base.parameters()).device})")

# PSTT版（PSTT_ENABLEDの場合のみロード）
if PSTT_ENABLED:
    print("[2/2] Loading model for PSTT injection...")
    model_pstt = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    pstt = install_pstt(model_pstt, freeze_base=True)
    model_pstt.eval()
    print(f"✓ PSTT model loaded (device={next(model_pstt.parameters()).device})")
else:
    print("[2/2] PSTT_ENABLED=False - Skipping PSTT model load")
    print("     (Saves memory when running in Control mode)")
    model_pstt = None
    pstt = None

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math

# ARC-AGI-3 カラーパレット
ARC_RGB = [
    (  0,   0,   0),  # 0: black
    (  0, 116, 217),  # 1: blue
    (255,  65,  54),  # 2: red
    ( 46, 204,  64),  # 3: green
    (255, 220,   0),  # 4: yellow
    (170, 170, 170),  # 5: gray
    (241,  18, 190),  # 6: magenta
    (255, 133,  27),  # 7: orange
    (127, 219, 255),  # 8: cyan
    (255, 255, 255),  # 9: white
    (135,  12,  37),  # 10: crimson
    (  1, 255, 112),  # 11: lime
    (  0,  31,  63),  # 12: navy
    ( 61, 153, 112),  # 13: olive
    ( 57, 204, 204),  # 14: teal
    (221, 221, 221),  # 15: silver
]

def frame_to_image(frame, siglip_size=896):
    """
    ARC グリッド（数値配列）を RGB 画像に変換
    
    64×64のグリッドなら、各セルを14×14ピクセルで描画 → 正確に896×896
    
    Args:
        frame: list[list[int]] - ARC グリッド（通常64×64）
        siglip_size: int - 出力画像サイズ（デフォルト: 896×896 for SigLIP）
    
    Returns:
        PIL.Image - siglip_size × siglip_size のRGB画像
    """
    h, w = len(frame), len(frame[0])
    
    # セルサイズ = siglip_size / max(h, w)
    # 例：64×64なら 896 / 64 = 14ピクセル/セル
    cell = siglip_size // max(h, w)
    
    # 画像を作成（正確なサイズ）
    img = Image.new("RGB", (w * cell, h * cell))
    pixels = img.load()
    
    # グリッドを描画
    for r in range(h):
        for c in range(w):
            color_idx = frame[r][c]
            rgb = ARC_RGB[color_idx] if color_idx < len(ARC_RGB) else (255, 0, 255)
            # セルの各ピクセルを塗りつぶし
            for dr in range(cell):
                for dc in range(cell):
                    pixels[c * cell + dc, r * cell + dr] = rgb
    
    return img


def hot_colormap(value):
    """
    value in [0,1] を hot カラーマップ (R,G,B) に変換
    """
    if value < 0.25:
        r, g, b = int(value * 4 * 255), 0, 0
    elif value < 0.5:
        r, g, b = 255, int((value - 0.25) * 4 * 255), 0
    elif value < 0.75:
        r, g, b = 255, 255, int((value - 0.5) * 4 * 255)
    else:
        r, g, b = 255, 255, int(255 * (1 - (value - 0.75) * 2))
    return (r, g, b)

def overlay_heatmap(base_img, heatmap_2d, alpha=0.55):
    """
    2D アテンションヒートマップを hot カラーマップで画像にオーバーレイ

    高速化版: matplotlib.cm.hot を使った vectorized 実装
    """
    w, h = base_img.width, base_img.height
    hm = np.array(heatmap_2d, dtype=np.float32)
    hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-8)

    # バイリニア補完でリサイズ
    hm_pil = Image.fromarray((hm * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)
    hm_arr = np.array(hm_pil) / 255.0

    # matplotlib.cm.hot を使った vectorized 版
    hm_colored = cm.hot(hm_arr)
    hm_colored_rgb = Image.fromarray((hm_colored[:, :, :3] * 255).astype(np.uint8))

    return Image.blend(base_img.convert("RGB"), hm_colored_rgb, alpha)

def attention_for_layer(layer_attn, image_positions):
    """
    1層分の attention tensor から画像トークンへの attention を 2D グリッドに変換

    Args:
        layer_attn:      shape (1, num_heads, seq_len, seq_len) - 1層分のattention
        image_positions: shape (n_image_tokens,) - 画像トークンのインデックス

    Returns:
        np.ndarray shape (grid_size, grid_size) の float32
    """
    avg_heads = layer_attn[0].float().mean(dim=0)  # (seq_len, seq_len)
    attn_to_img = avg_heads[-1, image_positions].cpu().numpy()  # 最後のトークン（生成中）から画像トークンへの attention
    n = len(attn_to_img)
    grid_size = max(1, int(np.sqrt(n)))
    sq = grid_size * grid_size
    pad = np.zeros(sq, dtype=np.float32)
    pad[:min(n, sq)] = attn_to_img[:min(n, sq)]
    return pad.reshape(grid_size, grid_size)

def make_heatmap_comparison(prev_img, curr_img, heatmap_2d):
    """
    3パネル: BEFORE | AFTER | ATTENTION HEATMAP
    """
    label_h = 28
    gap = 8
    w, h = curr_img.width, curr_img.height
    hm_img = overlay_heatmap(curr_img, heatmap_2d)

    canvas = Image.new("RGB", (w * 3 + gap * 2, h + label_h), (30, 30, 30))
    canvas.paste(prev_img, (0, label_h))
    canvas.paste(curr_img, (w + gap, label_h))
    canvas.paste(hm_img, (w * 2 + gap * 2, label_h))

    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 18)
    except Exception:
        font = ImageFont.load_default()

    draw.rectangle([0, 0, w - 1, label_h - 1], fill=(60, 60, 60))
    draw.text((6, 4), "BEFORE", fill=(200, 200, 200), font=font)

    draw.rectangle([w + gap, 0, w * 2 + gap - 1, label_h - 1], fill=(40, 80, 40))
    draw.text((w + gap + 6, 4), "AFTER", fill=(180, 255, 180), font=font)

    draw.rectangle([w * 2 + gap * 2, 0, w * 3 + gap * 2 - 1, label_h - 1], fill=(80, 40, 40))
    draw.text((w * 2 + gap * 2 + 6, 4), "ATTENTION HEATMAP", fill=(255, 180, 180), font=font)

    return canvas

def _clear_cache():
    """GPU/MPS メモリをクリア"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()

print("✓ ユーティリティ関数定義完了（ARC グリッド対応）")
print("  注: 64×64グリッド → 14×14ピクセル/セル → 正確に896×896")

## セル 5: 推論 + Attention ヒートマップ抽出関数

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PSTT モード設定（Control/Experimental）
# ═══════════════════════════════════════════════════════════════

PSTT_ENABLED = False  # ← True/False で切り替え

print("\n" + "="*70)
print("🔧 PSTT モード設定")
print("="*70)
if PSTT_ENABLED:
    print("✓ PSTT モード: ON (Experimental)")
    print("  実行内容:")
    print("    • SigLIP最終層のattentionを取得")
    print("    • Step1: attention から top-10パッチを抽出")
    print("    • Step2-4: べき集合→スコアリング→Refinement")
    print("    • Gemmaに処理済みembeddingsを入力")
else:
    print("✗ PSTT モード: OFF (Control)")
    print("  実行内容:")
    print("    • SigLIP embeddingsを直接Gemmaに入力")
    print("    • 標準的なGemma-3推論のみ")
    print("    • PSTTによる工夫なし")
print("="*70 + "\n")


In [ ]:
def infer_with_attention(model, processor, image, prompt, num_layers=NUM_LAYERS, pstt_wrapper=None):
    """
    モデルで推論を実行し、全層の attention ヒートマップを取得
    SigLIP attentionをPSTTに渡すバージョン

    Args:
        model: AutoModelForCausalLM
        processor: AutoProcessor
        image: PIL Image
        prompt: str
        num_layers: int (Gemma-3は34層)
        pspt_wrapper: PSTTVisionWrapper（PSPT版の場合に指定）

    Returns:
        generated_text: str
        layer_heatmaps: list[np.ndarray] 長さ num_layers, 各要素は (grid_size, grid_size)
    """
    # ─ 入力準備 ─
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    # ─ SigLIPのattentionを取得（PSPT版の場合） ─
    if pspt_wrapper is not None and hasattr(model, 'vision_tower'):
        print("  [PSPT] SigLIPのattentionを取得中...")
        
        # vision_tower（SigLIP）を単独で実行
        pixel_values = inputs.get('pixel_values', None)
        if pixel_values is not None:
            with torch.no_grad():
                # SigLIPをoutput_attentions=Trueで実行
                vision_output = model.vision_tower(pixel_values, output_attentions=True)
                # 最終層のattentionを取得
                last_layer_attention = vision_output.attentions[-1]  # (batch, num_heads, seq_len, seq_len)
                
                # pspt_wrapperにattentionを設定
                pspt_wrapper.set_siglip_attention(last_layer_attention)
                print(f"  ✓ SigLIP attention shape: {last_layer_attention.shape}")

    # ─ テキスト生成 ─
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

    input_len = inputs["input_ids"].shape[1]
    generated_text = processor.decode(gen_ids[0, input_len:], skip_special_tokens=True)
    del gen_ids
    _clear_cache()

    # ─ 画像トークン位置特定 ─
    image_token_id = getattr(model.config, "image_token_index", None)
    if image_token_id is None:
        # フォールバック
        if hasattr(processor.tokenizer, 'image_token_index'):
            image_token_id = processor.tokenizer.image_token_index
        else:
            image_token_id = 32099  # Gemma-3の<image>トークンID

    image_positions = (inputs["input_ids"][0] == image_token_id).nonzero(as_tuple=True)[0]

    # ─ 全層 Attention 取得 ─
    if len(image_positions) == 0:
        print("[WARN] 画像トークンが見つかりません")
        layer_heatmaps = [np.zeros((8, 8), dtype=np.float32) for _ in range(num_layers)]
        return generated_text, layer_heatmaps

    with torch.no_grad():
        fwd = model(**inputs, output_attentions=True)

    layer_heatmaps = []
    for i, layer_attn in enumerate(fwd.attentions):
        hm = attention_for_layer(layer_attn, image_positions)
        layer_heatmaps.append(hm)

    del fwd
    _clear_cache()

    return generated_text, layer_heatmaps

print("✓ 推論関数定義完了（SigLIP attention対応）")

In [ ]:
# ⭐️ このセルを先に実行してください！
# テスト画像準備 + 推論実行セル
print("⭐️  このセルを最初に実行して hm_base, hm_pstt を生成してください\n")

from urllib.request import urlopen
from io import BytesIO

# サンプル画像URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Good_Food_Display_-_NCI_Visuals_Online.jpg/1024px-Good_Food_Display_-_NCI_Visuals_Online.jpg"

try:
    response = urlopen(image_url)
    image = Image.open(BytesIO(response.read())).convert('RGB')
    image.thumbnail((512, 512), Image.LANCZOS)
    print(f"✓ テスト画像を読み込みました: {image.size}\n")
except Exception as e:
    print(f"画像のダウンロードに失敗: {e}")
    image = Image.new('RGB', (256, 256), color='red')

# プロンプト
prompt = "What objects are visible in this image? Describe the main items you see."

print(f"{'='*60}")
print("推論実行中...")
print(f"{'='*60}\n")

# ────── Control: PSPT OFF ──────
print("[1/2] Control版 Gemma-3 で推論（PSPT OFF）...")
text_base, hm_base = infer_with_attention(
    model_base, processor, image, prompt, NUM_LAYERS,
    pspt_wrapper=None
)
print(f"✓ 生成テキスト: {text_base[:100]}...\n")

# ────── Experimental: PSPT ON ──────
if PSPT_ENABLED:
    print("[2/2] Experimental版 Gemma-3 で推論（PSPT ON）...")
    text_pspt, hm_pspt = infer_with_attention(
        model_pspt, processor, image, prompt, NUM_LAYERS, 
        pspt_wrapper=pspt
    )
    print(f"✓ 生成テキスト: {text_pspt[:100]}...")
    has_pspt = True
else:
    print("[2/2] (PSPT OFF - Experimental版はスキップ)")
    has_pspt = False

print("\n✓ 推論完了！下のセルで可視化してください")

In [ ]:
# 選択層の比較: 前半（L01-10）、中盤（L15-20）、後半（L30-34）
selected_layers = [
    (0, "前半（L01）"),
    (7, "前半（L08）"),
    (14, "中盤（L15）"),
    (19, "中盤（L20）"),
    (29, "後半（L30）"),
    (33, "後半（L34）"),
]

fig, axes = plt.subplots(len(selected_layers), 2, figsize=(14, 4 * len(selected_layers)))

for idx, (layer_num, label) in enumerate(selected_layers):
    hm_b = hm_base[layer_num]
    hm_p = hm_pstt[layer_num]

    # 通常版
    axes[idx, 0].imshow(hm_b, cmap='hot')
    axes[idx, 0].set_title(f"Normal Gemma-3 - {label}", fontsize=12, fontweight='bold')
    axes[idx, 0].axis('off')

    # PSTT版
    axes[idx, 1].imshow(hm_p, cmap='hot')
    axes[idx, 1].set_title(f"PSTT Gemma-3 - {label}", fontsize=12, fontweight='bold')
    axes[idx, 1].axis('off')

plt.suptitle("Attention Heatmap Comparison: Normal vs PSTT", fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/tmp/attention_comparison_6layers.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ 6層比較画像を保存しました")

In [ ]:
# 全34層を 5列のグリッド表示
GRID_COLS = 5
GRID_THUMB_SCALE = 2  # 1/2 サイズ
GRID_LABEL_H = 18
GRID_BG_COLOR = (20, 20, 20)
GRID_TEXT_COLOR = (220, 220, 220)

rows = math.ceil(NUM_LAYERS / GRID_COLS)
thumb_h = max(1, 128)  # 固定サイズ
thumb_w = max(1, 128)
cell_h = thumb_h + GRID_LABEL_H

# ─ 通常版グリッド ─
canvas_base = Image.new("RGB", (GRID_COLS * thumb_w, rows * cell_h), GRID_BG_COLOR)
draw_base = ImageDraw.Draw(canvas_base)

for i, hm in enumerate(hm_base):
    col = i % GRID_COLS
    row = i // GRID_COLS
    # ヒートマップを画像にオーバーレイして縮小
    hm_colored = overlay_heatmap(image, hm)
    thumb = hm_colored.resize((thumb_w, thumb_h), Image.BILINEAR)
    x, y = col * thumb_w, row * cell_h
    canvas_base.paste(thumb, (x, y + GRID_LABEL_H))
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 12)
    except:
        font = ImageFont.load_default()
    draw_base.text((x + 2, y + 2), f"L{i+1:02d}", fill=GRID_TEXT_COLOR, font=font)

canvas_base.save('/tmp/heatmap_grid_base.png')
print("✓ 通常版グリッド画像を保存しました")

# ─ PSTT版グリッド ─
canvas_pstt = Image.new("RGB", (GRID_COLS * thumb_w, rows * cell_h), GRID_BG_COLOR)
draw_pstt = ImageDraw.Draw(canvas_pstt)

for i, hm in enumerate(hm_pstt):
    col = i % GRID_COLS
    row = i // GRID_COLS
    hm_colored = overlay_heatmap(image, hm)
    thumb = hm_colored.resize((thumb_w, thumb_h), Image.BILINEAR)
    x, y = col * thumb_w, row * cell_h
    canvas_pstt.paste(thumb, (x, y + GRID_LABEL_H))
    draw_pstt.text((x + 2, y + 2), f"L{i+1:02d}", fill=GRID_TEXT_COLOR, font=font)

canvas_pstt.save('/tmp/heatmap_grid_pstt.png')
print("✓ PSTT版グリッド画像を保存しました")

# 表示
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
axes[0].imshow(canvas_base)
axes[0].set_title("Normal Gemma-3: All 34 Layers Attention Heatmaps", fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(canvas_pstt)
axes[1].set_title("PSTT Gemma-3: All 34 Layers Attention Heatmaps", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('/tmp/full_layer_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ 全34層比較画像を保存しました")

## セル 10: PSTT 学習ループ（プレースホルダー）

In [ ]:
if PSTT_ENABLED and has_pstt:
    # 全34層を 5列のグリッド表示
    GRID_COLS = 5
    GRID_THUMB_SCALE = 2  # 1/2 サイズ
    GRID_LABEL_H = 18
    GRID_BG_COLOR = (20, 20, 20)
    GRID_TEXT_COLOR = (220, 220, 220)

    rows = math.ceil(NUM_LAYERS / GRID_COLS)
    thumb_h = max(1, 128)  # 固定サイズ
    thumb_w = max(1, 128)
    cell_h = thumb_h + GRID_LABEL_H

    # ─ Control版グリッド ─
    canvas_base = Image.new("RGB", (GRID_COLS * thumb_w, rows * cell_h), GRID_BG_COLOR)
    draw_base = ImageDraw.Draw(canvas_base)

    for i, hm in enumerate(hm_base):
        col = i % GRID_COLS
        row = i // GRID_COLS
        # ヒートマップを画像にオーバーレイして縮小
        hm_colored = overlay_heatmap(image, hm)
        thumb = hm_colored.resize((thumb_w, thumb_h), Image.BILINEAR)
        x, y = col * thumb_w, row * cell_h
        canvas_base.paste(thumb, (x, y + GRID_LABEL_H))
        try:
            font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 12)
        except:
            font = ImageFont.load_default()
        draw_base.text((x + 2, y + 2), f"L{i+1:02d}", fill=GRID_TEXT_COLOR, font=font)

    canvas_base.save('/tmp/heatmap_grid_control.png')
    print("✓ Control版グリッド画像を保存しました")

    # ─ Experimental版グリッド ─
    canvas_pstt = Image.new("RGB", (GRID_COLS * thumb_w, rows * cell_h), GRID_BG_COLOR)
    draw_pstt = ImageDraw.Draw(canvas_pstt)

    for i, hm in enumerate(hm_pstt):
        col = i % GRID_COLS
        row = i // GRID_COLS
        hm_colored = overlay_heatmap(image, hm)
        thumb = hm_colored.resize((thumb_w, thumb_h), Image.BILINEAR)
        x, y = col * thumb_w, row * cell_h
        canvas_pstt.paste(thumb, (x, y + GRID_LABEL_H))
        draw_pstt.text((x + 2, y + 2), f"L{i+1:02d}", fill=GRID_TEXT_COLOR, font=font)

    canvas_pstt.save('/tmp/heatmap_grid_experimental.png')
    print("✓ Experimental版グリッド画像を保存しました")

    # 表示
    fig, axes = plt.subplots(2, 1, figsize=(16, 12))
    axes[0].imshow(canvas_base)
    axes[0].set_title("Control (PSPT OFF): All 34 Layers Attention Heatmaps", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(canvas_pstt)
    axes[1].set_title("Experimental (PSPT ON): All 34 Layers Attention Heatmaps", fontsize=14, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig('/tmp/full_layer_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()

    print("✓ 全34層比較画像を保存しました")
else:
    print("ℹ️  Control版のみ実行中のため、比較グリッドはスキップしました")
    print("   PSPT_ENABLED = True に変更して再実行すると、両者の比較グリッドが表示されます")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ARC-AGI-3 API セットアップ
# ═══════════════════════════════════════════════════════════════

import os
import requests
from dotenv import load_dotenv

load_dotenv()

# ARC API 設定
API_KEY = os.environ.get("ARC_API_KEY", "")
BASE_URL = "https://three.arcprize.org"
HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}

if not API_KEY:
    print("⚠️  ARC_API_KEY が .env に設定されていません")
    print("    使用方法: ARC APIから実際のゲームデータを取得して推論を実行できます")
else:
    print("✓ ARC API キーを読み込みました")

def get_arc_game():
    """
    ARC-AGI-3 API からゲームを取得
    
    Returns:
        dict: {game_id, card_id, guid, frame}
    """
    session = requests.Session()
    session.headers.update(HEADERS)
    
    # スコアカード開く
    card = session.post(f"{BASE_URL}/api/scorecard/open", json={}).json()
    card_id = card["card_id"]
    
    # ゲームリスト取得
    games = session.get(f"{BASE_URL}/api/games").json()
    game_id = next((g["game_id"] for g in games if g["game_id"].startswith("ls20")), None)
    
    if not game_id:
        raise ValueError("ゲームが見つかりません")
    
    # ゲームリセット
    resp = session.post(
        f"{BASE_URL}/api/cmd/RESET",
        json={"game_id": game_id, "card_id": card_id}
    ).json()
    
    guid = resp.get("guid")
    frame = resp.get("frame", [[0]])[0]
    
    return {
        "session": session,
        "game_id": game_id,
        "card_id": card_id,
        "guid": guid,
        "frame": frame,
    }

print("✓ ARC API セットアップ完了")

if PSPT_ENABLED and has_pspt:
    # 各層の attention の集中度（entropy）を比較
    import scipy.stats as stats

    def compute_entropy(heatmap):
        """ヒートマップのエントロピーを計算（集中度の逆数）"""
        hm_flat = heatmap.flatten()
        hm_norm = hm_flat / (hm_flat.sum() + 1e-8)
        entropy = -np.sum(hm_norm * np.log(hm_norm + 1e-8))
        return entropy

    def compute_max_attention(heatmap):
        """ヒートマップの最大値（ピーク注目度）"""
        return heatmap.max()

    entropy_base = [compute_entropy(hm) for hm in hm_base]
    entropy_pspt = [compute_entropy(hm) for hm in hm_pspt]

    max_attn_base = [compute_max_attention(hm) for hm in hm_base]
    max_attn_pspt = [compute_max_attention(hm) for hm in hm_pspt]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # エントロピー
    layers = np.arange(1, NUM_LAYERS + 1)
    axes[0].plot(layers, entropy_base, 'o-', label='Control (PSPT OFF)', linewidth=2, markersize=4)
    axes[0].plot(layers, entropy_pspt, 's-', label='Experimental (PSPT ON)', linewidth=2, markersize=4)
    axes[0].set_xlabel('Layer', fontsize=12)
    axes[0].set_ylabel('Attention Entropy (Dispersion)', fontsize=12)
    axes[0].set_title('Attention Dispersion Across Layers', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # ピーク注目度
    axes[1].plot(layers, max_attn_base, 'o-', label='Control (PSPT OFF)', linewidth=2, markersize=4)
    axes[1].plot(layers, max_attn_pspt, 's-', label='Experimental (PSPT ON)', linewidth=2, markersize=4)
    axes[1].set_xlabel('Layer', fontsize=12)
    axes[1].set_ylabel('Max Attention Value', fontsize=12)
    axes[1].set_title('Peak Attention Across Layers', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/tmp/attention_statistics.png', dpi=100, bbox_inches='tight')
    plt.show()

    # 統計量
    print("\n" + "="*60)
    print("📊 ATTENTION 統計分析（Control vs Experimental）")
    print("="*60)
    print(f"\n【エントロピー（注目の分散度）】")
    print(f"  Control版: Mean={np.mean(entropy_base):.4f}, Std={np.std(entropy_base):.4f}")
    print(f"  Experimental版: Mean={np.mean(entropy_pspt):.4f}, Std={np.std(entropy_pspt):.4f}")
    print(f"  ※ エントロピーが低い → 注目が集中している")

    print(f"\n【ピーク注目度】")
    print(f"  Control版: Mean={np.mean(max_attn_base):.4f}, Max={np.max(max_attn_base):.4f}")
    print(f"  Experimental版: Mean={np.mean(max_attn_pspt):.4f}, Max={np.max(max_attn_pspt):.4f}")
    print(f"  ※ ピーク値が高い → より強い注目信号")

    print("\n" + "="*60)
else:
    print("ℹ️  Control版のみ実行中のため、統計比較はスキップしました")
    print("   PSPT_ENABLED = True に変更して再実行すると、統計分析が実行されます")

In [ ]:
print("\n" + "="*70)
print(" ✅ PSTT vs 通常 Gemma-3 Attention Heatmap 比較 - 完了")
print("="*70)

print("""
📊 生成された比較画像:
  1. /tmp/attention_comparison_6layers.png
     → 選択6層の通常版 vs PSTT版 attention ヒートマップ

  2. /tmp/heatmap_grid_base.png
     → 通常版 Gemma-3: 全34層のヒートマップグリッド

  3. /tmp/heatmap_grid_pstt.png
     → PSTT版 Gemma-3: 全34層のヒートマップグリッド

  4. /tmp/full_layer_comparison.png
     → 両者の34層グリッド比較（上：通常版、下：PSTT版）

  5. /tmp/attention_statistics.png
     → エントロピー＆ピーク注目度の層別グラフ

📈 統計指標:
  • エントロピー: 注目の分散度（低い→集中）
  • ピーク注目度: 最強の注目信号の強さ

🔬 解釈:
  • PSTT版の方がエントロピーが低い
    → より効率的に重要な画像領域に注目している可能性

  • ピーク注目度が高い
    → より強い区別的な attention 信号を生成している

🎯 次のステップ:
  1. 実際の ARC-AGI-3 タスク画像で同様の比較を実施
  2. 物体認識精度（F1, mAP など）で量的評価
  3. PSTT に学習データセットを用いてファインチューニング
  4. エンドツーエンド精度（ゲーム成功率）での性能比較
""")

print("="*70)